# Movie split and user EDA

Split feature movies 70 / 15 / 15 for a cold-start rating model, then look at users through their ratings of training movies only. A kept user has at least `MIN_USER_RATINGS` training ratings and at least `MIN_HIGH_RATINGS` scores of `HIGH_RATING` or higher. `genre_romance_high = 0.2` means 20% of the movies that user scored that high are tagged romance.

Nothing in this notebook is written to disk. Clustering still reads the existing `user_features.csv`.


In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
PROCESSED = Path("..") / "data" / "processed"
RANDOM_STATE = 42
CHUNKSIZE = 5_000_000

TRAIN_SIZE = 0.70
# The held-out 30% is split in half, so validation and test are 15% each.
HOLDOUT_TEST_SIZE = 0.50
POPULARITY_BINS = 3

MIN_USER_RATINGS = 10
MIN_HIGH_RATINGS = 5
HIGH_RATING = 4

N_SAMPLE_USERS = 50_000
MIN_GENRE_RATINGS = 5
SHRINK = 3


## 1. Load

Movie features come from `05_movie_features.ipynb`. Ratings are read in chunks with compact integer types. The `date` column is not needed.


In [ ]:
movies = pd.read_csv(PROCESSED / "movies_features.csv", converters={"cast_top5": ast.literal_eval})
GENRE_COLS = [column for column in movies.columns if column.startswith("genre_")]

chunks = pd.read_csv(
    PROCESSED / "ratings.csv",
    usecols=["netflix_movie_id", "customer_id", "rating"],
    dtype={"netflix_movie_id": "int32", "customer_id": "int32", "rating": "int8"},
    chunksize=CHUNKSIZE,
)
ratings = pd.concat(chunks, ignore_index=True)
n_all = len(ratings)
ratings = ratings[ratings["netflix_movie_id"].isin(movies["netflix_movie_id"])].reset_index(drop=True)

print(f"movies: {movies.shape}")
print(f"genre columns: {len(GENRE_COLS)}")
print(f"ratings: {len(ratings):,} (dropped {n_all - len(ratings):,} of movies without features)")
print(f"users: {ratings['customer_id'].nunique():,}")


## 2. Movie split

Movies are split 70 / 15 / 15, stratified so the three parts look alike on popularity (a tercile of the number of ratings). Documentaries form their own stratum, so some of them stay in training. The number of ratings is used only to stratify, not as a feature.


In [ ]:
movie_n = ratings.groupby("netflix_movie_id").size().rename("n_ratings")
movie_mean = ratings.groupby("netflix_movie_id")["rating"].mean().rename("avg_rating")
movies = movies.join(movie_n, on="netflix_movie_id").join(movie_mean, on="netflix_movie_id")
assert movies["n_ratings"].notna().all()
movies[["n_ratings", "avg_rating"]].describe().round(2)


In [ ]:
popularity = pd.qcut(movies["n_ratings"], POPULARITY_BINS, labels=["low", "mid", "high"])
strata = np.where(movies["genre_documentary"] == 1, "documentary", popularity.astype(str))
print("smallest strata:", pd.Series(strata).value_counts().tail(4).to_dict())

ids = movies["netflix_movie_id"].to_numpy()
train_ids, rest_ids, _, rest_strata = train_test_split(
    ids,
    strata,
    test_size=1 - TRAIN_SIZE,
    stratify=strata,
    random_state=RANDOM_STATE,
)
val_ids, test_ids = train_test_split(
    rest_ids,
    test_size=HOLDOUT_TEST_SIZE,
    stratify=rest_strata,
    random_state=RANDOM_STATE,
)
split_of = {
    **{movie_id: "train" for movie_id in train_ids},
    **{movie_id: "val" for movie_id in val_ids},
    **{movie_id: "test" for movie_id in test_ids},
}
movies["split"] = movies["netflix_movie_id"].map(split_of)
assert movies["split"].notna().all()
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))
movies["split"].value_counts()


**Balance check.** The three parts should have similar ratings, popularity, and genre mix. The average rating of validation and test movies is a check only, never a feature.


In [ ]:
balance = movies.groupby("split").agg(
    movies=("netflix_movie_id", "size"),
    ratings=("n_ratings", "sum"),
    median_n_ratings=("n_ratings", "median"),
    avg_rating=("avg_rating", "mean"),
    runtime=("runtime", "mean"),
    log_roi=("log_roi", "mean"),
    drama=("genre_drama", "mean"),
    comedy=("genre_comedy", "mean"),
    horror=("genre_horror", "mean"),
    documentaries=("genre_documentary", "sum"),
    us_only=("us_only", "mean"),
).loc[["train", "val", "test"]]
balance["pct_ratings"] = 100 * balance["ratings"] / balance["ratings"].sum()
balance.round(2)


## 3. Assign the split to ratings

Every rating gets the split of its movie. A lookup array indexed by movie id keeps the column at one byte per row.


In [ ]:
SPLITS = np.array(["train", "val", "test"])
split_lut = np.full(ratings["netflix_movie_id"].max() + 1, -1, dtype="int8")
for code_, name in enumerate(SPLITS):
    split_lut[movies.loc[movies["split"] == name, "netflix_movie_id"].to_numpy()] = code_
ratings["split"] = split_lut[ratings["netflix_movie_id"].to_numpy()]
assert (ratings["split"] >= 0).all()

train_ratings = ratings[ratings["split"] == 0]
print({name: f"{int((ratings['split'] == i).sum()):,}" for i, name in enumerate(SPLITS)})


## 4. Ratings per user on training movies

A profile uses ratings of training movies only. The table shows how many users each ratings floor would keep, and what share of the validation and test ratings belong to them. The histogram is that same count, binned.

The floors applied after the plot are `MIN_USER_RATINGS` and `MIN_HIGH_RATINGS` (scores of `HIGH_RATING` or higher). The printout is how many users each rule drops.


In [ ]:
train_n = train_ratings.groupby("customer_id").size()
eval_n = ratings[ratings["split"] > 0].groupby("customer_id").size()
n_users = ratings["customer_id"].nunique()

rows = []
for min_n in [5, 10, 20, 30, 50, 100]:
    kept = train_n.index[train_n >= min_n]
    rows.append(
        {
            "min_train_ratings": min_n,
            "users": len(kept),
            "pct_users": 100 * len(kept) / n_users,
            "pct_val_test_ratings": 100 * eval_n.reindex(kept).sum() / eval_n.sum(),
        }
    )
pd.DataFrame(rows).set_index("min_train_ratings").round(1)


In [ ]:
bins = [0, 4, 9, 19, 49, 99, 249, np.inf]
labels = ["1-4", "5-9", "10-19", "20-49", "50-99", "100-249", "250+"]
counts = pd.cut(train_n, bins=bins, labels=labels).value_counts().reindex(labels)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(labels, counts.to_numpy(), color="steelblue")
ax.set_xlabel("ratings of training movies per user")
ax.set_ylabel("users")
ax.set_title(f"Training ratings per user (median {train_n.median():.0f})")
fig.tight_layout()


In [ ]:
high_n = (
    train_ratings.loc[train_ratings["rating"] >= HIGH_RATING]
    .groupby("customer_id")
    .size()
    .reindex(train_n.index, fill_value=0)
)
fail_ratings = train_n < MIN_USER_RATINGS
fail_high = high_n < MIN_HIGH_RATINGS
eligible_users = train_n.index[~fail_ratings & ~fail_high]

print(f"Users with a training rating: {len(train_n):,}")
print(f"Dropped n_ratings < {MIN_USER_RATINGS}: {int(fail_ratings.sum()):,}")
print(f"Dropped n_high < {MIN_HIGH_RATINGS}: {int(fail_high.sum()):,}")
print(f"Dropped by both rules: {int((fail_ratings & fail_high).sum()):,}")
print(f"Kept: {len(eligible_users):,}")


## 5. User vs. movie: how much of a rating does each explain?

If users differ strongly in how generous they are, the user's mean rating is a useful model feature. Each rating is predicted by a simple rule and scored with $R^2$:

$$R^2 = 1 - \frac{\sum (\text{rating} - \text{prediction})^2}{\sum (\text{rating} - \text{average rating})^2}$$

$R^2$ is the share of rating variation the rule explains: 0 is no better than the overall average, and 1 is a perfect prediction. The three rules are the movie mean, the user mean, and the movie mean plus that user's average offset from the movies they rated.

This is computed on training ratings of a random sample of `N_SAMPLE_USERS` kept users. Every training rating of a sampled user is used.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_ids = rng.choice(
    eligible_users.to_numpy(),
    size=min(N_SAMPLE_USERS, len(eligible_users)),
    replace=False,
)
sample = train_ratings[train_ratings["customer_id"].isin(sample_ids)].copy()
sample["rating"] = sample["rating"].astype(float)

train_movie_mean = train_ratings.groupby("netflix_movie_id")["rating"].mean()
sample["movie_mean"] = sample["netflix_movie_id"].map(train_movie_mean)
sample["user_mean"] = sample.groupby("customer_id")["rating"].transform("mean")
user_offset = (sample["rating"] - sample["movie_mean"]).groupby(sample["customer_id"]).transform("mean")
sample["user_movie_bias"] = sample["movie_mean"] + user_offset

total_var = ((sample["rating"] - sample["rating"].mean()) ** 2).mean()
r2 = {
    name: 1 - ((sample["rating"] - sample[column]) ** 2).mean() / total_var
    for name, column in [
        ("movie mean", "movie_mean"),
        ("user mean", "user_mean"),
        ("user + movie bias", "user_movie_bias"),
    ]
}
print(f"sample: {len(sample_ids):,} users, {len(sample):,} ratings")
pd.Series(r2, name="R2").round(3)


## 6. Do users differ in genre taste?

Persona clustering only makes sense if users differ in which genres they like, beyond being generous or harsh overall. This section is a diagnostic. It does not build the `genre_*_high` columns.

**Genre score.** For each user and genre, and only for pairs with at least `MIN_GENRE_RATINGS` ratings:

$$\text{genre score} = \text{the user's average rating of that genre} - \text{the user's average rating of all films}$$

The score is a difference between two ratings, centered at 0. A user who averages 3.5 overall and 4.0 on action has an action score of +0.5. Subtracting the user's own average removes generosity: a harsh user and a generous user who both like action by half a star get the same score.

Spread alone is not evidence of taste. The scores are recomputed after shuffling each user's ratings across their own movies, which keeps their ratings and their movies but destroys the match between them. The spread ratio is the standard deviation of the real scores divided by the standard deviation of the shuffled scores. A ratio of 1 means users do not disagree about that genre beyond luck.

The histograms compare real scores (blue) with shuffled scores (grey) for the three genres with the highest ratio and the three with the lowest. A wider blue histogram means the genre divides users. A blue histogram shifted to the right means almost everyone rates that genre above their own average. The bar chart is the spread ratio of every genre. Bars near the red line at 1 are genres users do not really disagree about.


In [ ]:
def shuffle_within_user(user_ids, values, rng):
    '''Permute values inside each contiguous block of one user. user_ids must be sorted.'''
    block_start = np.r_[0, np.flatnonzero(np.diff(user_ids)) + 1, len(user_ids)]
    return np.concatenate(
        [rng.permutation(values[start:stop]) for start, stop in zip(block_start[:-1], block_start[1:])]
    )


def genre_score_table(sample, genre_matrix, min_genre_ratings):
    '''Mean within-user rating deviation by genre, real and after shuffling ratings.'''
    flags = genre_matrix.reindex(sample["netflix_movie_id"]).to_numpy()
    rows, scores = [], {}
    for column, flag in zip(genre_matrix.columns, flags.T):
        part = sample.loc[flag == 1, ["customer_id", "dev", "dev_shuffled"]]
        per_user = part.groupby("customer_id").agg(
            n=("dev", "size"),
            real=("dev", "mean"),
            shuffled=("dev_shuffled", "mean"),
        )
        per_user = per_user[per_user["n"] >= min_genre_ratings]
        if per_user.empty:
            continue
        name = column.removeprefix("genre_")
        scores[name] = per_user
        rows.append(
            {
                "genre": name,
                "users": len(per_user),
                "mean_dev": per_user["real"].mean(),
                "spread_real": per_user["real"].std(),
                "spread_shuffled": per_user["shuffled"].std(),
            }
        )
    taste = pd.DataFrame(rows).set_index("genre")
    taste["ratio"] = taste["spread_real"] / taste["spread_shuffled"]
    return taste.sort_values("ratio", ascending=False), scores


In [ ]:
sample = sample.sort_values("customer_id")
sample["dev"] = sample["rating"] - sample["user_mean"]
sample["dev_shuffled"] = shuffle_within_user(
    sample["customer_id"].to_numpy(),
    sample["dev"].to_numpy(),
    rng,
)
genre_matrix = movies.set_index("netflix_movie_id")[GENRE_COLS]
taste, genre_scores = genre_score_table(sample, genre_matrix, MIN_GENRE_RATINGS)
taste.round(3)


In [ ]:
def plot_genre_taste(taste, genre_scores):
    '''Histograms for the genres that divide users most and least, then the ratio bar chart.'''
    examples = list(taste.index[:3]) + list(taste.index[-3:])
    bins = np.linspace(-1.5, 1.5, 41)
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
    for ax, genre in zip(axes.ravel(), examples):
        scores = genre_scores[genre]
        pct = np.full(len(scores), 100 / len(scores))
        ax.hist(scores["real"], bins=bins, weights=pct, color="steelblue", alpha=0.7, label="real users")
        ax.hist(
            scores["shuffled"],
            bins=bins,
            weights=pct,
            histtype="step",
            color="gray",
            lw=1.5,
            label="shuffled (luck only)",
        )
        ax.axvline(0, color="black", lw=0.6)
        wider = 100 * (taste.loc[genre, "ratio"] - 1)
        ax.set_title(f"{genre}: real users {wider:.0f}% more spread than luck", fontsize=10)
    for ax in axes[:, 0]:
        ax.set_ylabel("% of users")
    for ax in axes[1]:
        ax.set_xlabel("genre score (stars above the user's own average)")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ordered = taste["ratio"].iloc[::-1]
    ax.barh(ordered.index, ordered.to_numpy(), color="steelblue")
    ax.axvline(1, color="#c44e52", ls="--", lw=1, label="1 = no more spread than luck")
    ax.set_xlabel("spread of real genre scores / spread under luck only")
    ax.set_title("How much more users disagree about each genre than luck would produce")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()


In [ ]:
plot_genre_taste(taste, genre_scores)


## 7. Rating distribution

The share of ratings at each star level, and how much that share varies between kept users, shows what a later threshold at `HIGH_RATING` means in practice.


In [ ]:
dist = ratings["rating"].value_counts(normalize=True).sort_index()
high_share = dist.loc[dist.index >= HIGH_RATING].sum()
print(f"share of ratings >= {HIGH_RATING}: {high_share:.1%} | = 5: {dist.loc[5]:.1%}")

user_ge4 = (
    (train_ratings["rating"] >= HIGH_RATING)
    .groupby(train_ratings["customer_id"])
    .mean()
    .reindex(eligible_users)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(dist.index.astype(str), 100 * dist.to_numpy(), color="steelblue")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("% of ratings")
axes[0].set_title("Rating distribution")
axes[1].hist(user_ge4, bins=40, color="steelblue", edgecolor="white")
axes[1].set_xlabel(f"share of the user's ratings that are >= {HIGH_RATING}")
axes[1].set_ylabel("users")
axes[1].set_title("Per-user share of high ratings (kept users)")
fig.tight_layout()


In [ ]:
user_ge4.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).rename(f"share of ratings >= {HIGH_RATING}")


**How to read this.** The left panel is the share of all ratings at each star. The right panel, and the quantiles above, show how often a kept user gives a high score. Users mostly rate films they chose, and they differ in how often those ratings clear `HIGH_RATING`. A single cutoff on a predicted rating would recommend most movies to generous users and few to harsh ones.


## 8. Train-only movie statistics

`avg_rating`, `rating_std`, and `n_ratings` are computed from ratings of training movies only. They are not written to disk.


In [ ]:
movie_stats = (
    train_ratings.groupby("netflix_movie_id")["rating"]
    .agg(avg_rating="mean", rating_std="std", n_ratings="size")
    .reset_index()
)
movie_stats.describe().round(3)


**Director and cast track records.** The score is the average `avg_rating` of a person's other films, shrunk toward the global training mean with strength `SHRINK`, so one lucky film cannot produce an extreme score:

$$\text{track record} = \frac{\text{sum of avg ratings of the other films} + m \cdot \text{global mean}}{\text{number of other films} + m}$$

Only training films count as other films. A training movie leaves itself out. A validation or test movie uses all of that person's training films, because its own rating is never known. `cast_track_record` averages the first 5 billed actors.


In [ ]:
def shrunk_track_record(pairs, train_avg, shrink, global_mean):
    '''Shrunk mean of each person's other training films.

    pairs is one row per (netflix_movie_id, person). A movie missing from
    train_avg is validation or test, so every training film counts as other.
    '''
    own = pairs["netflix_movie_id"].map(train_avg)
    totals = (
        pd.DataFrame({"person": pairs["person"], "own": own})
        .dropna()
        .groupby("person")["own"]
        .agg(["sum", "count"])
    )
    joined = pairs.join(totals, on="person").fillna({"sum": 0, "count": 0})
    other_sum = joined["sum"] - own.fillna(0)
    other_n = joined["count"] - own.notna().astype(int)
    return (other_sum + shrink * global_mean) / (other_n + shrink)


In [ ]:
train_avg = movie_stats.set_index("netflix_movie_id")["avg_rating"]
global_mean = train_avg.mean()

director_pairs = movies[["netflix_movie_id", "director"]].rename(columns={"director": "person"})
director_tr = shrunk_track_record(director_pairs, train_avg, SHRINK, global_mean)

cast_pairs = (
    movies[["netflix_movie_id", "cast_top5"]]
    .explode("cast_top5")
    .rename(columns={"cast_top5": "person"})
    .dropna(subset=["person"])
    .reset_index(drop=True)
)
cast_tr = shrunk_track_record(cast_pairs, train_avg, SHRINK, global_mean)
cast_mean = cast_tr.groupby(cast_pairs["netflix_movie_id"]).mean()

track_records = pd.DataFrame(
    {
        "netflix_movie_id": movies["netflix_movie_id"],
        "director_track_record": director_tr.to_numpy(),
        "cast_track_record": movies["netflix_movie_id"].map(cast_mean).fillna(global_mean).to_numpy(),
    }
)
track_records.describe().round(3)


Spearman correlation of the track records with the movie's average rating. On validation and test movies this is the cold-start check: those movies' own ratings were not used to build the score.


In [ ]:
check = track_records.merge(movies[["netflix_movie_id", "split", "avg_rating"]], on="netflix_movie_id")
spearman = pd.DataFrame(
    {
        split: {
            column: part[column].corr(part["avg_rating"], method="spearman")
            for column in ["director_track_record", "cast_track_record"]
        }
        for split, part in check.groupby("split")
    }
).T.loc[["train", "val", "test"]]
spearman.round(3)


## 9. User statistics

These are computed for kept users from ratings of training movies only, and they are not written to disk.

| column | meaning |
|---|---|
| `user_mean` | the user's average rating: how generous they are |
| `user_std` | standard deviation of the user's ratings: whether they use the whole 1–5 scale |
| `user_n` | number of training movies the user rated |
| `user_pct_ge4` | share of the user's ratings that are at least `HIGH_RATING` |

`user_pct_ge4` is related to `user_mean` but not the same. A user who rates 3, 3, 3, 5 and a user who rates 3, 4, 4, 3 both average 3.5, while the first clears the high-score bar on 25% of films and the second on 50%.


In [ ]:
eligible_train = train_ratings[train_ratings["customer_id"].isin(eligible_users)]
user_stats = eligible_train.groupby("customer_id")["rating"].agg(
    user_mean="mean",
    user_std="std",
    user_n="size",
)
user_stats["user_pct_ge4"] = (
    eligible_train["rating"] >= HIGH_RATING
).groupby(eligible_train["customer_id"]).mean()
user_stats = user_stats.reset_index()
user_stats.describe().round(3)


## 10. Liked-genre shares

`genre_romance_high = 0.2` means 20% of the movies this user scored `HIGH_RATING` or higher are tagged romance. The value is the mean of that genre's 0/1 flag over those training movies. A movie with several genres counts in each share, so a user's shares can sum to more than 1. The denominator is the number of high-scored movies, not the number of tags.


In [ ]:
def liked_genre_shares(ratings, movies, users, genre_cols, high_rating):
    '''Share of each user's high-rated training movies that carry each genre tag.'''
    high = ratings.loc[
        ratings["customer_id"].isin(users) & (ratings["rating"] >= high_rating),
        ["customer_id", "netflix_movie_id"],
    ]
    flags = movies.set_index("netflix_movie_id")[genre_cols]
    tagged = high.join(flags, on="netflix_movie_id")
    shares = tagged.groupby("customer_id", sort=False)[genre_cols].mean()
    shares.columns = [f"{column}_high" for column in genre_cols]
    return shares


In [ ]:
genre_shares = liked_genre_shares(train_ratings, movies, eligible_users, GENRE_COLS, HIGH_RATING)
print(f"users: {len(genre_shares):,}")
genre_shares.head()


In [ ]:
genre_shares.sum(axis=1).describe().round(2).rename("sum of genre shares")


## 11. Liked-genre profile

The bar is the mean `genre_*_high` across kept users. `genre_entropy` is the entropy of those shares after they are scaled to sum to 1. Because movies carry more than one genre, entropy summarizes the tag mix, not a share of movies. The heatmap shows which liked-genre shares move together.


In [ ]:
means = genre_shares.mean().sort_values()
labels = [name.removeprefix("genre_").removesuffix("_high") for name in means.index]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(labels, means.to_numpy(), color="steelblue")
ax.set_xlabel(f"mean share of movies scored >= {HIGH_RATING}")
ax.set_title("Genres in movies users score high")
fig.tight_layout()


In [ ]:
def genre_share_entropy(shares):
    '''Entropy of genre shares after scaling each user to sum to 1.'''
    mass = shares.sum(axis=1)
    scaled = shares.div(mass.where(mass > 0), axis=0).to_numpy(dtype="float64")
    log_share = np.zeros_like(scaled)
    positive = scaled > 0
    np.log(scaled, where=positive, out=log_share)
    entropy = np.full(len(shares), np.nan)
    has_tags = mass.to_numpy() > 0
    entropy[has_tags] = -(scaled[has_tags] * log_share[has_tags]).sum(axis=1)
    return pd.Series(entropy, index=shares.index, name="genre_entropy")


In [ ]:
genre_entropy = genre_share_entropy(genre_shares)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(genre_entropy.dropna(), bins=40, color="steelblue")
ax.set_xlabel("genre_entropy")
ax.set_ylabel("users")
ax.set_title("Tag mix of movies scored high")
fig.tight_layout()


In [ ]:
genre_entropy.describe().round(3)


In [ ]:
genre_corr = genre_shares.corr()
labels = [name.removeprefix("genre_").removesuffix("_high") for name in genre_corr.columns]
fig, ax = plt.subplots(figsize=(8, 6.5))
image = ax.imshow(genre_corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(labels)), labels, rotation=90)
ax.set_yticks(range(len(labels)), labels)
ax.set_title("Correlation of liked-genre shares")
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()


## 12. Leakage checks

Movie statistics use training movies only. Kept users meet both floors. Track records cover every feature movie. Liked-genre shares stay between 0 and 1. Nothing is written. `07_user_clustering.ipynb` still reads the existing `user_features.csv`.


In [ ]:
train_movie_ids = set(movies.loc[movies["split"] == "train", "netflix_movie_id"])
assert set(movie_stats["netflix_movie_id"]) <= train_movie_ids
assert user_stats["user_n"].min() >= MIN_USER_RATINGS
assert high_n.reindex(user_stats["customer_id"]).min() >= MIN_HIGH_RATINGS
assert len(track_records) == len(movies) and track_records.notna().all().all()
assert set(genre_shares.index) == set(eligible_users)
assert genre_shares.min().min() >= 0 and genre_shares.max().max() <= 1

print(f"movie_stats: {movie_stats.shape}")
print(f"track_records: {track_records.shape}")
print(f"user_stats: {user_stats.shape}")
print(f"genre_shares: {genre_shares.shape}")
print("Nothing was written. Clustering still reads data/processed/user_features.csv.")
